# Demo of 3D Generative Model Latent Disentanglement via Local Eigenprojection

To run the demo download the `demo_files` folder and copy it in the project directory.

---

### Network initialisation


In [ ]:
%matplotlib notebook

import os
import trimesh
import torch
import numpy as np
# from torch_geometric.data import Data

import utils
from model_manager import ModelManager

# demo_directory = "demo_files"
demo_directory = "./outputs/experiments/320/" # 324
configurations = utils.get_config(os.path.join(demo_directory, "config.yaml"))

if 'combined' in configurations['data']['dataset_type']:
    norm_dict_path = "./precomputed/norm_babies_faces_combined.pt"
else:
    norm_dict_path = "./precomputed/norm_babies_faces.pt"

if configurations['model']['age_disentanglement']:

    if configurations['model']['age_per_feature'] == True:
        no_remainder = configurations['model']['latent_size'] % configurations['model']['age_latent_size'] == 0
        correct_value = configurations['model']['latent_size'] // configurations['model']['age_latent_size'] == 5
        assert no_remainder and correct_value
        
configurations['model']['latent_size'] += configurations['model']['age_latent_size']


if not torch.cuda.is_available():
    device = torch.device("cpu")
    print("GPU not available, running on CPU")
else:
    device = torch.device("cuda")

manager = ModelManager(
    configurations=configurations, device=device,
    precomputed_storage_path=configurations['data']['precomputed_path'])
manager.resume(os.path.join(demo_directory, "checkpoints"))

normalization_dict = torch.load(norm_dict_path)

Resume from epoch 600


### Randomly generate a shape and create a scene

Every time you run this cell, a new shape is generated. Run it untli you are satisfied with the generated shape.

In [7]:
import pickle
from torch_geometric.data import Data

In [9]:
# assign if we are using pre defined meshes like relatives data
relatives = False # False

# if true, then get the mesh, load it and run it through the encoder to get z then generate it using the decoder
if relatives:

    # Load the transformations from the .pkl file
    if 'combined' in configurations['data']['dataset_type']:
        pkl_name = 'precomputed/transforms_babies_faces_combined.pkl'
    else:
        pkl_name = 'precomputed/transforms_babies_faces.pkl'

    if configurations['model']['age_disentanglement']:
        with open(pkl_name, 'rb') as f:     
            pre_transform = pickle.load(f)

    mesh_path = "/raid/compass/athena/data/relatives/combined/20241102_albie_2.obj" #20241018_oliver.obj"
    mesh = trimesh.load_mesh(mesh_path)
    mesh_verts = torch.tensor(mesh.vertices, dtype=torch.float)

    # normalize the mesh
    mesh_verts = (mesh_verts - normalization_dict['mean']) / normalization_dict['std']

    # reshape mesh_verts to go from torch.Size([16085, 3]) to torch.Size([1, 16085, 3])
    mesh_verts = mesh_verts.unsqueeze(0)

    print("no1")
    print(mesh_verts.size())

    ## torch.Size([16, 16085, 3])
    # data = Data(x=mesh_verts)

    # print("no2")
    # print(data.x.size())

    # # Apply pre_transform if it exists
    # if pre_transform is not None:
    #     data = pre_transform(data)

    # print(data.x.size())
    
    z = manager.encode(mesh_verts.to(device))
    gen_verts = manager.generate(z.to(device))[0, :, :]
    print("no2")
    print(z.size())
    gen_verts = gen_verts * normalization_dict['std'].to(device) + \
        normalization_dict['mean'].to(device)
    print("no3")
    print(gen_verts.size())
else:
    age_latent_size = configurations['model']['age_latent_size']
    feature_latent_size = configurations['model']['latent_size'] - age_latent_size

    z_features = torch.randn([1, feature_latent_size])

    single_random_value = torch.randn(1)
    z_age = torch.full([1, age_latent_size], single_random_value.item())

    z = torch.cat((z_features, z_age), dim=1)

    gen_verts = manager.generate(z.to(device))[0, :, :] 
    gen_verts = gen_verts * normalization_dict['std'].to(device) + \
        normalization_dict['mean'].to(device)

In [10]:
from pythreejs import *
from IPython.display import display
import ipywidgets

view_width = 800
view_height = 800

# z = torch.randn([1, manager.model_latent_size])

# age_latent_size = configurations['model']['age_latent_size']
# feature_latent_size = configurations['model']['latent_size'] - age_latent_size

# z_features = torch.randn([1, feature_latent_size])

# single_random_value = torch.randn(1)
# z_age = torch.full([1, age_latent_size], single_random_value.item())

# z = torch.cat((z_features, z_age), dim=1)

# # print(normalization_dict['std'].to(device).size())
# # print(normalization_dict['mean'].to(device).size())

# gen_verts = manager.generate(z.to(device))[0, :, :] 
# gen_verts = gen_verts * normalization_dict['std'].to(device) + \
#     normalization_dict['mean'].to(device)
initially_gen_verts = gen_verts.clone()
faces = manager.template.face

def compute_vertex_colours(current_verts):
    source_dist = manager.compute_vertex_errors(current_verts, initially_gen_verts)
    colours = utils.errors_to_colors(source_dist.unsqueeze(dim=0), min_value=0,
                                     max_value=10, cmap='plasma') / 255
    return colours.squeeze().cpu().detach().numpy()


buffer_verts = BufferAttribute(gen_verts.detach().cpu().numpy().tolist(), normalized=False)
buffer_faces = BufferAttribute(np.uint32(faces.t().numpy().tolist()).ravel(), normalized=False)
buffer_colours = BufferAttribute(compute_vertex_colours(gen_verts).tolist(), normalized=False)

geometry = BufferGeometry(
    attributes={
        'position': buffer_verts,
        'index': buffer_faces,
        'color': buffer_colours
    })
geometry.exec_three_obj_method('computeVertexNormals')

material = MeshPhongMaterial(color="#34eb46", specular="#222222", shininess=15)
material_colours = MeshPhongMaterial(specular="#222222", shininess=15, vertexColors='VertexColors')

mesh = Mesh(geometry, material=material)
mesh_colours = Mesh(geometry, material=material_colours)

camera = PerspectiveCamera(position=[2, 0, 3], aspect=view_width/view_height)
ambient_light = AmbientLight(intensity=0.2)
ambient_light_dispmap = AmbientLight(intensity=1)
key_light = SpotLight(position=[0, 10, 10], angle = 0.3, penumbra = 0.1)

key_light.target = mesh
mesh.castShadow = True
mesh.receiveShadow = True
mesh_colours.castShadow = True
mesh_colours.receiveShadow = True

scene = Scene(children=[mesh, camera, key_light, ambient_light])
scene_colours = Scene(children=[mesh_colours, camera, ambient_light_dispmap])

controller = OrbitControls(controlling=camera)
renderer = Renderer(camera=camera, scene=scene, controls=[controller],
                    width=view_width, height=view_height, antialias=True)
renderer_colours = Renderer(camera=camera, scene=scene_colours, controls=[controller],
                            width=view_width/3, height=view_height/3, antialias=True)
renderers_pair = ipywidgets.HBox([renderer_colours, renderer])
renderers_pair.layout.align_items = "center"
display(renderers_pair)

### Create GUI

In [14]:
def update_vertices(z_i_value, z_i_index):
    
    age_latent_index = configurations['model']['latent_size'] - configurations['model']['age_latent_size']
    data_type = configurations['data']['dataset_type'].split("_", 1)[1]

    import pickle
    storage_path = os.path.join('precomputed', f'normalise_age_{data_type}.pkl')
    with open(storage_path, 'rb') as file:
        age_train_mean, age_train_std = \
            pickle.load(file)

    if z_i_index >= age_latent_index:
        z_i_value = (z_i_value.new - age_train_mean) / age_train_std
    else:
        z_i_value = z_i_value.new

    z[0, z_i_index] = z_i_value
    verts = manager.generate(z.to(device))[0, :, :] 
    verts = verts * normalization_dict['std'].to(device) + \
        normalization_dict['mean'].to(device)
    colours = compute_vertex_colours(verts)
    verts = verts.detach().cpu().numpy()
    v = verts.astype("float32", copy=False)
    geometry.attributes["position"].array = v
    geometry.attributes["position"].needsUpdate = True
    mesh.geometry.exec_three_obj_method('computeVertexNormals')
    geometry.attributes["color"].array = colours.astype("float32", copy=False)
    geometry.attributes["color"].needsUpdate = True
    mesh.geometry.verticesNeedUpdate = True
    mesh.geometry.elementsNeedUpdate = True
    mesh.geometry.colorsNeedUpdate = True
    mesh.exec_three_obj_method('update')
    mesh_colours.geometry.verticesNeedUpdate = True
    mesh_colours.geometry.elementsNeedUpdate = True
    mesh_colours.geometry.colorsNeedUpdate = True
    mesh_colours.exec_three_obj_method('update')
    controller.exec_three_obj_method('update')
    camera.exec_three_obj_method('updateProjectionMatrix')
    scene.exec_three_obj_method('update')
    scene_colours.exec_three_obj_method('update')
    
if configurations['model']['age_latent_size'] == 9:
    if 'combined' in configurations['data']['dataset_type']:
        # 9 features combined dataset
        color2name_dict = {'[161 241 250 255]': "temporal", '[ 57 129 136 255]': "eyes",  '[250 156   0 255]': "cheekbones", 
                        '[250 231 144 255]': "cheeks", '[169  95 250 255]': "jaw", '[136 250 184 255]': "forehead", 
                        '[250 174 203 255]': "chin",  '[250  57 159 255]': "mouth", '[250 129  96 255]': "nose"}
    else:
        # 9 features non-combined dataset
        color2name_dict = {'[ 57 130 135 255]': "eyes", '[160 241 251 255]': "temporal", '[135 251 185 255]': "forehead",
                        '[251 155   0 255]': "cheekbones", '[168  95 251 255]': "jaw", '[251 130  96 255]': "nose", 
                        '[251 231 144 255]': "cheeks", '[251  57 158 255]': "mouth", '[251 174 204 255]': "chin" }
else:
    print("Error: color2name_dict not defined")



region_sliders_names = []
region_sliders = []
grouped_region_indices = []
for r_name, r_range in manager.latent_regions.items():
    region_indices = list(range(r_range[0], r_range[1]))
    grouped_region_indices.append(region_indices)
    sl = []
    for i in region_indices:
        current_s = ipywidgets.FloatSlider(min=-3, max=3, step=0.05, value=z[0, i], description=f"z_{str(i)}")
        current_s.observe(lambda x, y=i: update_vertices(x, y), names='value')
        sl.append(current_s)
    region_sliders.append(ipywidgets.VBox(sl))
    region_sliders_names.append(color2name_dict[r_name])

display(region_sliders)

[VBox(children=(FloatSlider(value=0.0012351949699223042, description='z_0', max=3.0, min=-3.0, step=0.05), FloatSlider(value=1.086828589439392, description='z_1', max=3.0, min=-3.0, step=0.05), FloatSlider(value=-0.27195078134536743, description='z_2', max=3.0, min=-3.0, step=0.05), FloatSlider(value=-0.8555039763450623, description='z_3', max=3.0, min=-3.0, step=0.05), FloatSlider(value=1.117714285850525, description='z_4', max=3.0, min=-3.0, step=0.05), FloatSlider(value=1.0798356533050537, description='z_5', max=3.0, min=-3.0, step=0.05))),

This code create a seperate section of the sliders on the GUI for all of the age latents that correspond to each feature. It uses the min and max of the normalised ages. 

In [15]:
total_latents = z.shape[1]
extra_latents_start = r_range[1]
extra_latents_indices = list(range(extra_latents_start, total_latents))

features = ["temporal", "eyes", "cheekbones", "cheeks", "jaw", "forehead", "chin", "mouth", "nose"]

extra_sliders = []

import pickle
# storage_path = os.path.join(demo_directory, 'z_stats.pkl')
# with open(storage_path, 'rb') as file:
#     z_stats = pickle.load(file)

# age_latent_size = configurations['model']['age_latent_size']

# z_age_mins = z_stats['mins'][-age_latent_size:]
# z_age_maxs = z_stats['maxs'][-age_latent_size:]


data_type = configurations['data']['dataset_type'].split("_", 1)[1]

storage_path = os.path.join('precomputed', f'normalise_age_{data_type}.pkl')
with open(storage_path, 'rb') as file:
    age_train_mean, age_train_std = \
        pickle.load(file)
    
unnormalized_value = (z[0, extra_latents_indices] * age_train_std) + age_train_mean

###############

for i in range(len(extra_latents_indices)):

    current_s = ipywidgets.FloatSlider(min=0, max=17, step=1, value=unnormalized_value[i], description=f"{features[i]}")
    # current_s = ipywidgets.FloatSlider(min=z_age_mins[i], max=z_age_maxs[i], step=0.05, value=z[0, extra_latents_indices[i]], description=f"{features[i]}")
    current_s.observe(lambda x, y=extra_latents_indices[i]: update_vertices(x, y), names='value')
    print(current_s)
    extra_sliders.append(current_s)

region_sliders.append(ipywidgets.VBox(extra_sliders))
region_sliders_names.append("Age Features")  

###################

# # Create individual sliders for each feature's age
# extra_sliders = []
# for i in range(len(extra_latents_indices)):
#     current_s = ipywidgets.FloatSlider(
#         min=0, max=17, step=1, value=unnormalized_value[i].item(), description=f"{features[i]}"
#     )
#     current_s.observe(lambda change, idx=extra_latents_indices[i]: update_vertices(change, idx), names='value')
#     extra_sliders.append(current_s)

# # Create a master slider to control all feature age sliders
# master_slider = ipywidgets.FloatSlider(
#     min=0, max=17, step=1, value=0, description='All Features Age'
# )

# # Function to update all feature sliders based on the master slider
# def update_all_sliders(change):
#     for slider in extra_sliders:
#         slider.value = change['new']

# # Link the master slider to the update function
# master_slider.observe(update_all_sliders, names='value')

# # Create a VBox to hold all the sliders
# region_sliders = [ipywidgets.VBox([master_slider] + extra_sliders)]
# region_sliders_names = ["Age Features"]

# # Display the sliders
display(region_sliders)


[VBox(children=(FloatSlider(value=0.0012351949699223042, description='z_0', max=3.0, min=-3.0, step=0.05), FloatSlider(value=1.086828589439392, description='z_1', max=3.0, min=-3.0, step=0.05), FloatSlider(value=-0.27195078134536743, description='z_2', max=3.0, min=-3.0, step=0.05), FloatSlider(value=-0.8555039763450623, description='z_3', max=3.0, min=-3.0, step=0.05), FloatSlider(value=1.117714285850525, description='z_4', max=3.0, min=-3.0, step=0.05), FloatSlider(value=1.0798356533050537, description='z_5', max=3.0, min=-3.0, step=0.05))),
 VBox()]

In [16]:
out_mesh_dir = os.path.join(demo_directory, "out_meshes")
if not os.path.isdir(out_mesh_dir):
    os.mkdir(out_mesh_dir)

fname_widget = ipywidgets.Text(placeholder="mesh_name.ply", description="Filename:", disabled=False)
out_message_widget = ipywidgets.Output()


def compute_vertex_colours(current_verts):
    source_dist = manager.compute_vertex_errors(current_verts, initially_gen_verts)
    colours = utils.errors_to_colors(source_dist.unsqueeze(dim=0), min_value=0,
                                     max_value=10, cmap='plasma') / 255
    return colours.squeeze().cpu().detach().numpy()


def save_current_mesh(b):
    verts = manager.generate(z.to(device))[0, ::] 
    verts = verts * normalization_dict['std'].to(device) + \
        normalization_dict['mean'].to(device)
    v_col = compute_vertex_colours(verts)
    mesh = trimesh.Trimesh(
        verts.cpu().detach().numpy(),
        manager.template.face.t().cpu().numpy(),
        vertex_colors=v_col)
    
    fname = fname_widget.value
    if fname.endswith(".ply") or fname.endswith(".ply"):
        mesh.export(os.path.join(out_mesh_dir, fname))
        with out_message_widget:
            print("Mesh saved!")
    else:
        with out_message_widget:
            print(f"'{fname}' is not a valid meshfile name. Make sure it finishes in '.ply' or '.obj'")

    
save_button_widget = ipywidgets.Button(description="Save", disabled=False, button_style='info')
save_button_widget.on_click(save_current_mesh)
saving_widgets = ipywidgets.HBox([fname_widget, save_button_widget, out_message_widget])

add 'random' buttom to randomly generate a new face mesh 

In [8]:
def reset_mesh_and_sliders(b=None):
    # Generate a new random z vector
    # global z
    z = torch.randn([1, manager.model_latent_size]) #.to(device)
    
    # # Update sliders for each region
    # for i, slider_group in enumerate(region_sliders):
    #     for j, slider in enumerate(slider_group.children):
    #         z_index = grouped_region_indices[i][j]
    #         slider.value = z[0, z_index].item()
    
    # # Update extra sliders for age features
    # for i, slider in enumerate(extra_sliders):
    #     z_index = extra_latents_indices[i]
    #     slider.value = z[0, z_index].item()
    
    # Regenerate the mesh with the new z vector
    # update_vertices(z[0, 0], 0)  # Trigger mesh update using the first slider as a proxy
    for i in range(z.shape[1]):
        update_vertices(z[0, i], i)

# Create the "Random" button
random_button = ipywidgets.Button(description="Random mesh", button_style='success')
random_button.on_click(reset_mesh_and_sliders)


# Add the "Random" button to the UI layout
random_widget = ipywidgets.HBox([random_button])

### Run the GUI 

Each slider corresponds to a latent variable. When sliders are changed, the VAE generates a new shapes that is displayed in real time. Sliders are grouped according to the anatomical region that they are influencing. On the left side a displacement map shows the regions that were altered while manipulating the latent variables with the sliders. Displacements are computed from the random mesh that was initially generated. When you are satisfied with the final result, you can also save the mesh.

In [17]:
accordion = ipywidgets.Accordion(children=region_sliders, titles=region_sliders_names)
ipywidgets.VBox([ipywidgets.HBox([renderers_pair, accordion]), saving_widgets])#, random_widget])